
# Customer Churn Prediction

## Project Overview

This project uses historical telecom customer data to predict whether a customer is likely to churn.

The workflow covers:

- Data loading and inspection
- Data cleaning and preprocessing
- Exploratory data analysis
- Feature encoding and scaling
- Train-test splitting
- Logistic Regression
- Random Forest Classification
- Model evaluation using accuracy, precision, recall, F1-score, and confusion matrices
- Feature importance analysis

> **Note:** The models in this notebook are baseline models. The focus is on building and evaluating a clear end-to-end classification workflow.



## 1. Imports


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

pd.set_option("display.max_columns", None)



## 2. Load the Dataset

The project uses the **Telco Customer Churn Dataset**.

The target variable is `Churn`.


In [ ]:

DATA_PATH = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())


In [ ]:

print("Data types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False).head(10))

print("\nDuplicate rows:", df.duplicated().sum())



## 3. Data Cleaning

The dataset contains `TotalCharges` as a field that should be treated as numeric.

`customerID` is an identifier rather than a predictive feature, so it is removed before modelling.

Rows with missing values created by numeric conversion are removed.


In [ ]:

# Convert TotalCharges to numeric.
# Invalid/blank values become NaN and are handled below.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Remove identifier column
df = df.drop(columns=["customerID"])

# Remove rows with missing values
df = df.dropna().reset_index(drop=True)

print("Shape after cleaning:", df.shape)
print("Remaining missing values:", df.isnull().sum().sum())



## 4. Exploratory Data Analysis

Before modelling, we examine the distribution of churn and selected customer characteristics.


In [ ]:

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Churn")
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.show()

print(df["Churn"].value_counts(normalize=True).mul(100).round(2))


In [ ]:

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("Monthly Charges vs Churn")
plt.xlabel("Churn")
plt.ylabel("Monthly Charges")
plt.show()


In [ ]:

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="Churn", y="tenure")
plt.title("Tenure vs Churn")
plt.xlabel("Churn")
plt.ylabel("Tenure (months)")
plt.show()


In [ ]:

contract_churn = pd.crosstab(
    df["Contract"],
    df["Churn"],
    normalize="index"
).mul(100).round(2)

display(contract_churn)

contract_churn.plot(kind="bar", stacked=True, figsize=(9, 5))
plt.title("Churn Distribution by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Percentage of Customers")
plt.legend(title="Churn")
plt.xticks(rotation=0)
plt.show()



## 5. Prepare Features and Target

The target variable is converted into binary form:

- `No` → 0
- `Yes` → 1

Categorical predictors are one-hot encoded.

The data is split into training and test sets **before scaling** so that information from the test set does not influence the training transformation.


In [ ]:

# Encode target
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

X = df.drop(columns=["Churn"])
y = df["Churn"]

# One-hot encode categorical variables
X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)



## 6. Feature Scaling

Standardization is fitted only on the training data and then applied to the test data.

This avoids test-set information leaking into the preprocessing stage.


In [ ]:

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



## 7. Logistic Regression

Logistic Regression is used as a baseline classification model.

It estimates the probability of a customer belonging to the churn class.


In [ ]:

log_reg = LogisticRegression(max_iter=1000, random_state=42)

log_reg.fit(X_train_scaled, y_train)

y_pred_lr = log_reg.predict(X_test_scaled)

print("Logistic Regression Classification Report")
print(classification_report(y_test, y_pred_lr))


In [ ]:

lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr, zero_division=0)
lr_recall = recall_score(y_test, y_pred_lr, zero_division=0)
lr_f1 = f1_score(y_test, y_pred_lr, zero_division=0)

print(f"Accuracy : {lr_accuracy:.4f}")
print(f"Precision: {lr_precision:.4f}")
print(f"Recall   : {lr_recall:.4f}")
print(f"F1-score : {lr_f1:.4f}")


In [ ]:

cm_lr = confusion_matrix(y_test, y_pred_lr)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm_lr,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"]
)
plt.title("Logistic Regression - Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()



## 8. Random Forest Classifier

Random Forest is used as a second baseline model.

It combines multiple decision trees and can capture non-linear relationships between customer characteristics and churn.


In [ ]:

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Random Forest Classification Report")
print(classification_report(y_test, y_pred_rf))


In [ ]:

rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf, zero_division=0)
rf_recall = recall_score(y_test, y_pred_rf, zero_division=0)
rf_f1 = f1_score(y_test, y_pred_rf, zero_division=0)

print(f"Accuracy : {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall   : {rf_recall:.4f}")
print(f"F1-score : {rf_f1:.4f}")


In [ ]:

cm_rf = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm_rf,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"]
)
plt.title("Random Forest - Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()



## 9. Model Comparison

Accuracy alone does not fully describe a churn classifier. In particular, **recall for the churn class** is useful because it measures how many of the customers who actually churned were identified by the model.


In [ ]:

results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [lr_accuracy, rf_accuracy],
    "Precision": [lr_precision, rf_precision],
    "Recall": [lr_recall, rf_recall],
    "F1-score": [lr_f1, rf_f1]
})

display(results.round(4))


In [ ]:

metrics = ["Accuracy", "Precision", "Recall", "F1-score"]

results.set_index("Model")[metrics].plot(
    kind="bar",
    figsize=(10, 5)
)

plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.show()



## 10. Random Forest Feature Importance

Feature importance provides an indication of which encoded features contributed most strongly to the Random Forest's predictions.

This should be interpreted as model-specific importance rather than proof of causality.


In [ ]:

feature_importance = pd.Series(
    rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

display(feature_importance.head(15))


In [ ]:

plt.figure(figsize=(9, 6))
feature_importance.head(15).sort_values().plot(kind="barh")
plt.title("Top 15 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()



## 11. Key Observations

Based on the exploratory analysis and baseline models:

- Customers with higher monthly charges show greater churn tendency in the dataset.
- Customers with lower tenure show greater churn tendency.
- Contract type is associated with different churn patterns.
- The two baseline classifiers provide a useful starting point for churn prediction.
- Churn-class recall should be considered alongside overall accuracy when evaluating the usefulness of the model.

## 12. Limitations and Future Improvements

The current notebook focuses on baseline classification models. Future work could explore:

- Class-balancing methods such as SMOTE.
- Hyperparameter tuning.
- Gradient Boosting or XGBoost.
- Additional feature engineering.
- Decision-threshold optimization to improve churn recall.
- Model calibration and more extensive validation.

The current results should therefore be treated as a baseline rather than a production-ready churn prediction system.



## 13. Conclusion

This project demonstrates an end-to-end machine learning workflow for telecom customer churn prediction, covering data cleaning, exploratory analysis, feature preparation, classification modelling, and evaluation.

The main modelling objective is not only to achieve high overall accuracy, but also to understand how effectively the models identify customers who actually churn.
